# E5 -- Fashion-MNIST x ResNet-20 (fills empty grid cell)

Adds the ResNet-20/Fashion-MNIST cell to the architecture x dataset grid. Uses the **same protocol as the existing CIFAR-10/ResNet-20 cell** (SGD lr=0.1, momentum=0.9, Nesterov, wd=1e-3, MultiStepLR; no augmentation; CE 200ep -> MSE 600ep).

**Input adaptation**: Fashion-MNIST is 28x28 grayscale, but ResNet-20 (as used in the existing CIFAR-10 cell) expects 32x32 RGB. We resize to 32x32 and replicate the grayscale channel three times. This keeps the architecture identical to the CIFAR-10 cell -- the only difference between the two cells is the dataset.

**Why 10 classes (no protocol issues)**: Unlike CIFAR-100, Fashion-MNIST is 10-class so the canonical two-phase MSE protocol works (CE->MSE switch did not cause the kind of catastrophic feature collapse we saw on 100 classes).

**Outputs**: `fmnist_resnet20_s{0,1,2}.csv` and `fmnist_resnet20_summary.csv`.

In [1]:

import torch, torchvision, time, os, sys
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
import torchvision.transforms as T

torch.backends.cudnn.benchmark        = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Runs on both Kaggle and Colab. Output dir is auto-detected.
if os.path.isdir('/kaggle/working'):
    PLATFORM = 'kaggle'
    SAVE_DIR = '/kaggle/working/'
    DATA_DIR = '/kaggle/working/data/'
elif 'google.colab' in sys.modules or os.path.isdir('/content'):
    PLATFORM = 'colab'
    # Mount Google Drive so per-seed CSVs survive a Colab session disconnect.
    # If mounting fails (e.g. permission denied), fall back to /content/ and
    # warn the user that the run is no longer crash-safe.
    DRIVE_DIR = '/content/drive/MyDrive/NC_revision/'
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        SAVE_DIR = DRIVE_DIR
        print(f'Drive mounted - outputs will be saved to {SAVE_DIR}')
    except Exception as exc:
        print(f'WARNING: Drive mount failed ({exc}). Falling back to /content/.')
        print('         A mid-run disconnect WILL lose progress.')
        SAVE_DIR = '/content/'
    # Datasets stay on local Colab disk so they do not eat Drive quota.
    DATA_DIR = '/content/data/'
else:
    PLATFORM = 'local'
    SAVE_DIR = './'
    DATA_DIR = './data/'
os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
assert torch.cuda.is_available(), \
    'No GPU - Runtime -> Change runtime type -> A100 (Colab) or enable GPU (Kaggle)'
print(f'Platform: {PLATFORM}')
print(f'SAVE_DIR: {SAVE_DIR}')
print(f'GPU:      {torch.cuda.get_device_name(0)}')
print(f'Torch:    {torch.__version__}')

Mounted at /content/drive
Drive mounted - outputs will be saved to /content/drive/MyDrive/NC_revision/
Platform: colab
SAVE_DIR: /content/drive/MyDrive/NC_revision/
GPU:      NVIDIA A100-SXM4-80GB
Torch:    2.11.0+cu128


In [2]:

# Fashion-MNIST resized to 32x32 RGB so the ResNet-20 architecture stays
# identical to the existing CIFAR-10/ResNet-20 cell. Only the dataset
# differs between this run and the published CIFAR-10/ResNet-20 row.
transform = T.Compose([
    T.Resize((32, 32)),               # 28x28 -> 32x32 (bilinear by default)
    T.Grayscale(num_output_channels=3),  # 1 channel -> 3 channels (replicate)
    T.ToTensor(),
    T.Normalize((0.2860, 0.2860, 0.2860), (0.3530, 0.3530, 0.3530)),
])
trainset = torchvision.datasets.FashionMNIST(DATA_DIR,
    train=True,  download=True, transform=transform)
testset  = torchvision.datasets.FashionMNIST(DATA_DIR,
    train=False, download=True, transform=transform)
train_loader = DataLoader(trainset, batch_size=128, shuffle=True,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
test_loader  = DataLoader(testset,  batch_size=256, shuffle=False,
                          num_workers=4, persistent_workers=True,
                          prefetch_factor=2, pin_memory=True)
print(f'Fashion-MNIST (resized to 32x32 RGB): '
      f'{len(trainset):,} train / {len(testset):,} test')
print(f'Train batches: {len(train_loader)} x 128')


100%|██████████| 26.4M/26.4M [00:02<00:00, 10.6MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 163kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.19MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 17.9MB/s]

Fashion-MNIST (resized to 32x32 RGB): 60,000 train / 10,000 test
Train batches: 469 x 128


In [3]:

class BasicBlock(nn.Module):
    def __init__(self, in_c, out_c, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_c, out_c, 3, stride=stride,
                               padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_c)
        self.conv2 = nn.Conv2d(out_c, out_c, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_c)
        self.skip  = nn.Sequential()
        if stride != 1 or in_c != out_c:
            self.skip = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_c))
    def forward(self, x):
        return F.relu(self.bn2(self.conv2(F.relu(self.bn1(self.conv1(x)))))
                      + self.skip(x))

class ResNet20(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1  = nn.Conv2d(3, 16, 3, padding=1, bias=False)
        self.bn1    = nn.BatchNorm2d(16)
        self.layer1 = self._make(16, 16, 3, 1)
        self.layer2 = self._make(16, 32, 3, 2)
        self.layer3 = self._make(32, 64, 3, 2)
        self.pool   = nn.AdaptiveAvgPool2d(1)
        self.fc     = nn.Linear(64, num_classes)
        self._feats = None
        self.pool.register_forward_hook(
            lambda m, i, o: setattr(self, '_feats', o.flatten(1).detach()))
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out')
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def _make(self, in_c, out_c, n, stride):
        layers = [BasicBlock(in_c, out_c, stride)]
        for _ in range(n - 1):
            layers.append(BasicBlock(out_c, out_c, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.layer3(self.layer2(self.layer1(x)))
        return self.fc(self.pool(x).flatten(1))

    def get_features(self, x):
        self(x); return self._feats

    def get_classifier_weights(self):
        return self.fc.weight.detach()

m = ResNet20(num_classes=10).to(DEVICE)
x = torch.randn(4, 3, 32, 32).to(DEVICE)
print(f'ResNet-20 (100-class): feats={tuple(m.get_features(x).shape)}, '
      f'params={sum(p.numel() for p in m.parameters())/1e6:.3f}M')
del m, x

ResNet-20 (100-class): feats=(4, 64), params=0.272M


In [4]:

@torch.no_grad()
def compute_nc(model, loader, K):
    model.eval()
    fl, ll = [], []
    for x, y in loader:
        fl.append(model.get_features(x.to(DEVICE, non_blocking=True)).cpu())
        ll.append(y)
    H = torch.cat(fl).float(); Y = torch.cat(ll)
    mu_G = H.mean(0)
    mu_c = torch.stack([H[Y==c].mean(0) for c in range(K)])
    M    = mu_c - mu_G
    Sw   = sum((H[Y==c]-mu_c[c]).T @ (H[Y==c]-mu_c[c])
               for c in range(K)) / len(H)
    Sb   = M.T @ M / K
    nc1  = (torch.trace(Sw) / torch.trace(Sb).clamp(1e-10)).item()
    Mn   = F.normalize(M, dim=1)
    cos  = Mn @ Mn.T
    mask = ~torch.eye(K, dtype=torch.bool)
    nc2  = (cos[mask] - (-1./(K-1))).abs().mean().item()
    Wn   = F.normalize(model.get_classifier_weights().cpu(), dim=1)
    nc3  = (1 - (Mn * Wn).sum(1).mean()).item()
    return {'nc1': nc1, 'nc2': nc2, 'nc3': nc3,
            'feat_norm': H.norm(dim=1).mean().item()}

def evaluate(model, loader):
    model.eval(); correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            correct += (model(x).argmax(1) == y).sum().item()
            total   += len(y)
    return correct / total

print('NC metrics + evaluate ready.')

NC metrics + evaluate ready.


In [5]:

# CIFAR-100 terminal-phase threshold is relaxed to 0.90 because, without
# data augmentation, ResNet-20 needs more than the 200 CE epochs of Phase 1
# to reach 99% training accuracy on 100 classes. 0.90 still corresponds to
# good fit and is documented in the response-to-reviewers letter.
TERMINAL_ACC = 0.90

def run_fmnist_resnet(model, name, lr=0.1, wd=1e-3,
                 phase1=200, phase2=600, nc_every=10, K=10):
    model = model.to(DEVICE)
    rows = []; terminal = False
    t_nc_strict = None; fn_at_strict = None
    t_nc_relaxed = None; fn_at_relaxed = None
    t0 = time.time()

    for phase, loss_fn, n_ep, milestones in [
        (1, 'ce',  phase1, [100, 150]),
        (2, 'mse', phase2, [300, 450]),
    ]:
        opt = torch.optim.SGD(model.parameters(), lr=lr,
                              momentum=0.9, weight_decay=wd, nesterov=True)
        sch = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=milestones,
                                                   gamma=0.1)
        off = phase1 if phase == 2 else 0
        for ep_l in range(1, n_ep + 1):
            ep = off + ep_l
            model.train()
            for x, y in train_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                logits = model(x)
                if loss_fn == 'mse':
                    loss = F.mse_loss(logits, F.one_hot(y, K).float())
                else:
                    loss = F.cross_entropy(logits, y)
                loss.backward(); opt.step()
            sch.step()

            if ep_l % nc_every == 0 or ep_l == n_ep:
                tr = evaluate(model, train_loader)
                te = evaluate(model, test_loader)
                if tr >= TERMINAL_ACC and not terminal:
                    terminal = True
                    print(f'  [{name}] Terminal phase at epoch {ep} '
                          f'(train acc >= {TERMINAL_ACC})')
                # Always compute NC in Phase 2 even if terminal was never
                # reached in Phase 1 - guarantees a non-empty CSV.
                if terminal or phase == 2:
                    nc = compute_nc(model, train_loader, K)
                else:
                    nc = {'nc1': None, 'nc2': None,
                          'nc3': None, 'feat_norm': None}
                rows.append({'epoch': ep, 'phase': phase,
                             'train': tr, 'test': te, **nc})
                if nc['nc1'] is not None:
                    print(f'  ep={ep:>4} tr={tr:.4f} te={te:.4f} '
                          f'nc1={nc["nc1"]:.5f} fn={nc["feat_norm"]:.3f} '
                          f't={(time.time()-t0)/60:.1f}m')
                    if t_nc_relaxed is None and nc['nc1'] < 0.05:
                        t_nc_relaxed = ep; fn_at_relaxed = nc['feat_norm']
                        print(f'    *** NC1<0.05 at ep {ep} '
                              f'fn={fn_at_relaxed:.4f}')
                    if t_nc_strict is None and nc['nc1'] < 0.01:
                        t_nc_strict = ep; fn_at_strict = nc['feat_norm']
                        print(f'    *** NC1<0.01 at ep {ep} '
                              f'fn={fn_at_strict:.4f}')

    print(f'  [{name}] Done in {(time.time()-t0)/60:.1f} min')
    return (pd.DataFrame(rows),
            t_nc_strict, fn_at_strict,
            t_nc_relaxed, fn_at_relaxed)

print('run_fmnist_resnet ready.')

run_fmnist_resnet ready.


In [6]:

fmnist_resnet20_results = []
for seed in range(3):
    print(f'\n=== Fashion-MNIST ResNet-20  seed={seed} ===')
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    model = ResNet20(num_classes=10)
    df, ts, fs, tr_, fr = run_fmnist_resnet(model, f'fmnistresnet-s{seed}',
                                       lr=0.1, wd=1e-3,
                                       phase1=200, phase2=600)
    df.to_csv(f'{SAVE_DIR}fmnist_resnet20_s{seed}.csv', index=False)
    fmnist_resnet20_results.append({'seed': seed,
                             'T_NC_strict': ts, 'fn_strict': fs,
                             'T_NC_relaxed': tr_, 'fn_relaxed': fr,
                             'test_acc_final': df.test.iloc[-1]})
    pd.DataFrame(fmnist_resnet20_results).to_csv(
        f'{SAVE_DIR}fmnist_resnet20_summary.csv', index=False)
    print(f'  saved fmnist_resnet20_s{seed}.csv and fmnist_resnet20_summary.csv')

summary = pd.DataFrame(fmnist_resnet20_results)
print('\n=== CIFAR-100 summary ===')
print(summary.to_string(index=False))

ok = summary.dropna(subset=['fn_strict'])
if len(ok) >= 2:
    fns = ok['fn_strict'].values
    print(f'\nfn at NC1<0.01 (N={len(ok)}): '
          f'mean={fns.mean():.4f} std={fns.std():.4f} '
          f'CV={100*fns.std()/fns.mean():.1f}%')


=== Fashion-MNIST ResNet-20  seed=0 ===
  [fmnistresnet-s0] Terminal phase at epoch 20 (train acc >= 0.9)
  ep=  20 tr=0.9304 te=0.9135 nc1=0.18439 fn=5.876 t=2.9m
  ep=  30 tr=0.8678 te=0.8556 nc1=0.18751 fn=5.670 t=4.1m
  ep=  40 tr=0.9139 te=0.9004 nc1=0.17036 fn=6.325 t=5.4m
  ep=  50 tr=0.9052 te=0.8934 nc1=0.17009 fn=5.510 t=6.7m
  ep=  60 tr=0.9201 te=0.9031 nc1=0.18213 fn=5.232 t=8.0m
  ep=  70 tr=0.8670 te=0.8582 nc1=0.18903 fn=4.901 t=9.2m
  ep=  80 tr=0.9118 te=0.8982 nc1=0.17436 fn=5.938 t=10.5m
  ep=  90 tr=0.8999 te=0.8842 nc1=0.17000 fn=5.683 t=11.8m
  ep= 100 tr=0.9112 te=0.8974 nc1=0.18411 fn=5.175 t=13.1m
  ep= 110 tr=0.9863 te=0.9296 nc1=0.12720 fn=6.288 t=14.3m
  ep= 120 tr=0.9891 te=0.9266 nc1=0.12887 fn=6.182 t=15.6m
  ep= 130 tr=0.9889 te=0.9279 nc1=0.12017 fn=6.401 t=16.8m
  ep= 140 tr=0.9819 te=0.9202 nc1=0.11971 fn=6.141 t=18.1m
  ep= 150 tr=0.9850 te=0.9204 nc1=0.12875 fn=5.868 t=19.4m
  ep= 160 tr=1.0000 te=0.9343 nc1=0.06980 fn=6.197 t=20.6m
  ep= 170 tr=1

In [7]:

out_files = [f'{SAVE_DIR}fmnist_resnet20_summary.csv'] + \
            [f'{SAVE_DIR}fmnist_resnet20_s{s}.csv' for s in range(3)]
if PLATFORM == 'colab' and SAVE_DIR.startswith('/content/drive'):
    print('Files saved to Drive:', SAVE_DIR)
    for fp in out_files:
        if os.path.exists(fp):
            print(' ', fp, f'({os.path.getsize(fp)} bytes)')
elif PLATFORM == 'colab':
    from google.colab import files
    for fp in out_files:
        if os.path.exists(fp):
            files.download(fp)
else:
    print('Files saved in', SAVE_DIR)
    for fp in out_files:
        if os.path.exists(fp):
            print(' ', fp, f'({os.path.getsize(fp)} bytes)')

Files saved to Drive: /content/drive/MyDrive/NC_revision/
  /content/drive/MyDrive/NC_revision/fmnist_resnet20_summary.csv (229 bytes)
  /content/drive/MyDrive/NC_revision/fmnist_resnet20_s0.csv (8530 bytes)
  /content/drive/MyDrive/NC_revision/fmnist_resnet20_s1.csv (8605 bytes)
  /content/drive/MyDrive/NC_revision/fmnist_resnet20_s2.csv (8433 bytes)
